# Puntos 7 y 9 — SentinelGrafo

Este notebook desarrolla el **punto 7 (Análisis de Resultados)** y el **punto 9 (Conclusiones)** a partir de la ejecución de `SentinelGrafo_Local.ipynb`.

El enfoque no se limita a una demostración académica de GNN, sino que se orienta a evaluar en qué medida el software puede **apoyar la resolución de una problemática real**: detectar y clasificar mensajes críticos en plataformas digitales para facilitar monitoreo, priorización y respuesta.

Las evidencias visuales y métricas se exportaron automáticamente a la carpeta `06_Recursos/evidencias_sentinelgrafo`.


## 7. Análisis de Resultados

La evaluación se realizó sobre dos casos de estudio:

- **Caso 1:** clasificación binaria de *Disaster Tweets*.
- **Caso 2:** clasificación multiclase de *AG News*.

En ambos casos se compararon tres enfoques: **Baseline con Logistic Regression sobre TF-IDF**, **GraphSAGE** y **GCN**. Las métricas reportadas corresponden al conjunto de prueba.

Aunque los datasets empleados tienen una finalidad experimental, ambos representan escenarios cercanos a una necesidad concreta: **filtrar grandes volúmenes de texto para separar información prioritaria de información no prioritaria**. Bajo esa lógica, el análisis no solo evalúa cuál modelo obtiene mejor puntuación, sino cuál ofrece mejores condiciones para incorporarse a un software inteligente orientado a la toma de decisiones.


### 7.1 Caso 1 — Disaster Tweets

| Modelo | Accuracy | Precision | Recall | F1-macro |
|---|---:|---:|---:|---:|
| Baseline (Logistic Regression) | 0.7876 | 0.7920 | 0.7876 | 0.7868 |
| GraphSAGE | 0.7746 | 0.7772 | 0.7746 | 0.7741 |
| GCN | 0.7571 | 0.7635 | 0.7570 | 0.7556 |

En este primer caso, el mejor resultado lo obtuvo el **baseline clásico**. GraphSAGE quedó ligeramente por debajo, mientras que GCN mostró el rendimiento más bajo de los tres modelos. Esto sugiere que, para una tarea binaria de texto corto como tweets, la representación TF-IDF combinada con un clasificador lineal ya captura gran parte de la señal discriminativa.

Desde una lectura comparativa, el uso del grafo **sí permitió aprender estructura relacional entre mensajes similares**, pero esa estructura no fue suficiente para superar la frontera de decisión del baseline. En otras palabras, el grafo aportó contexto, pero no añadió una ganancia cuantitativa clara frente al costo adicional del modelo GNN.

Llevado al problema real de SentinelGrafo, esto significa que el software **sí puede servir como filtro inicial para apoyar vigilancia digital y detección de publicaciones potencialmente críticas**, pero todavía conviene priorizar el modelo que entregue más aciertos operativos con menor complejidad. En este experimento, ese rol lo cumple mejor el baseline, especialmente si se piensa en un despliegue rápido para monitoreo de alertas o incidentes reportados por usuarios.


**Curvas de entrenamiento**

![Curvas de entrenamiento](../06_Recursos/evidencias_sentinelgrafo/curvas_entrenamiento_1.png)

En la fila superior de la figura se observa el comportamiento del Caso 1. En ambos modelos GNN la *train loss* disminuye de manera sostenida, pero la *test loss* deja de mejorar relativamente pronto y luego empieza a crecer. Ese patrón es consistente con **sobreajuste moderado**. El criterio de *early stopping* evitó que el deterioro fuera mayor, deteniendo a GraphSAGE en la época 37 y a GCN en la 38.

También se aprecia que **GraphSAGE alcanza una accuracy de prueba más estable** que GCN. GCN, aunque aprende rápido al inicio, presenta una curva de validación más irregular y termina con menor capacidad de generalización.


**Matrices de confusión**

![Matrices de confusión](../06_Recursos/evidencias_sentinelgrafo/matrices_confusion_1.png)

En la fila superior de la matriz de confusión se aprecia que ambos modelos reconocen razonablemente bien la clase **"No desastre"**, pero tienen más dificultad para recuperar correctamente la clase **"Desastre real"**. GraphSAGE obtuvo 475 verdaderos positivos para esa clase, mientras que GCN obtuvo 444.

Esto indica un sesgo práctico importante: el sistema tiende a ser algo más conservador al identificar eventos realmente críticos. En aplicaciones reales, este comportamiento puede traducirse en **falsos negativos** que reducen la utilidad operativa del modelo si el objetivo principal es no dejar pasar alertas relevantes.

Por tanto, desde la perspectiva de solución, el criterio relevante no es solo la métrica agregada, sino la capacidad del sistema para **minimizar omisiones en mensajes que podrían requerir atención inmediata**. Un software de apoyo a monitoreo ciudadano, gestión de emergencias o análisis de riesgo reputacional necesita justamente ese equilibrio entre precisión y sensibilidad.


**Proyección t-SNE de embeddings**

![t-SNE de embeddings](../06_Recursos/evidencias_sentinelgrafo/proyeccion_tsne_1.png)

En la fila superior del t-SNE se observa que las dos clases forman regiones parcialmente diferenciadas, aunque todavía existe un grado visible de mezcla en zonas intermedias. La proyección de **GraphSAGE** muestra agrupaciones algo más coherentes, mientras que **GCN** presenta una distribución más entrelazada y con mayor dispersión de errores.

Desde el punto de vista representacional, esto respalda la idea de que **GraphSAGE construyó embeddings más útiles que GCN** para este problema, aunque la mejora visual no alcanzó para superar al baseline en las métricas finales.


### 7.2 Caso 2 — AG News

| Modelo | Accuracy | Precision | Recall | F1-macro |
|---|---:|---:|---:|---:|
| Baseline (Logistic Regression) | 0.8915 | 0.8909 | 0.8915 | 0.8911 |
| GraphSAGE | 0.8773 | 0.8771 | 0.8773 | 0.8770 |
| GCN | 0.8627 | 0.8624 | 0.8627 | 0.8624 |

En el segundo caso se repite la misma tendencia general: **el baseline vuelve a ser el mejor modelo**, seguido por GraphSAGE y luego GCN. Sin embargo, la diferencia entre baseline y GraphSAGE es relativamente pequeña, lo que muestra que la representación gráfica fue competitiva incluso en un problema multiclase más grande.

AG News contiene categorías con rasgos léxicos relativamente marcados, por lo que un enfoque TF-IDF + Logistic Regression tiene una ventaja natural. Por eso, aunque la GNN modela relaciones entre noticias cercanas en el espacio semántico, el clasificador lineal sigue aprovechando de forma muy eficiente la señal textual superficial.

Este segundo caso es valioso porque amplía la lectura del software más allá del escenario de desastre. Muestra que SentinelGrafo también puede entenderse como una base para **triage automático de información textual** en contextos donde se requiere organizar contenido por tipo, urgencia o temática antes de que un analista humano intervenga.


**Curvas de entrenamiento**

![Curvas de entrenamiento](../06_Recursos/evidencias_sentinelgrafo/curvas_entrenamiento_1.png)

En la fila inferior de la figura se aprecia que los dos modelos mejoran rápidamente durante las primeras épocas y luego entran en una fase de estabilización. El comportamiento de **GraphSAGE** es más limpio: la accuracy de prueba se mantiene alrededor de 0.87 y la *test loss* se estabiliza antes del *early stopping*. En cambio, **GCN** se queda en una meseta más baja, cercana a 0.86.

Otra observación importante es que, aun cuando la *train loss* continúa descendiendo, la mejora de generalización se vuelve marginal. Esto refuerza la necesidad del *early stopping* y sugiere que la capacidad del modelo ya capturó la mayor parte del patrón útil antes de llegar al máximo de 200 épocas.


**Matrices de confusión**

![Matrices de confusión](../06_Recursos/evidencias_sentinelgrafo/matrices_confusion_1.png)

En la fila inferior se observa que la clase **Sports** es la más fácil de clasificar para ambas arquitecturas. GraphSAGE logra 1147 aciertos sobre 1200, y GCN 1129. Las clases con mayor dificultad son **Business** y **Sci/Tech**, donde aparecen más cruces entre categorías semánticamente próximas.

Ese patrón es consistente con la naturaleza del dominio: noticias de negocios y tecnología suelen compartir vocabulario, temas de innovación, mercado y empresas. Por ello, incluso con estructura de grafo, las fronteras entre ambas clases siguen siendo menos nítidas que en Sports.

En una aplicación real, esta observación importa porque muestra que el software puede responder bien cuando las categorías están claramente diferenciadas, pero requerirá mejores representaciones semánticas cuando las clases compartan vocabulario o contexto. Esa es una condición frecuente en sistemas de clasificación para soporte operativo, donde muchas alertas no vienen separadas de forma limpia.


**Proyección t-SNE de embeddings**

![t-SNE de embeddings](../06_Recursos/evidencias_sentinelgrafo/proyeccion_tsne_1.png)

En la fila inferior del t-SNE se distinguen cuatro regiones principales asociadas a las clases de AG News. En **GraphSAGE** los clústeres aparecen más compactos y relativamente mejor separados, especialmente para **World** y **Sports**. En **GCN** la estructura global también es reconocible, pero con mayor superposición en la zona central, donde se mezclan varios errores de **Business** y **Sci/Tech**.

Esto confirma que la construcción del grafo sí permitió una organización semántica útil en el espacio de embeddings. No obstante, esa organización todavía no supera la simplicidad y eficacia del baseline clásico en términos de métricas finales.


### 7.3 Comparativa global

![Comparativa global](../06_Recursos/evidencias_sentinelgrafo/comparativa_global_1.png)

La comparación conjunta permite extraer tres hallazgos principales:

1. **El baseline fue el mejor modelo en ambos datasets.**
   En Disaster Tweets alcanzó un F1-macro de 0.7868, por encima de GraphSAGE (0.7741) y GCN (0.7556). En AG News también lideró con 0.8911 frente a 0.8770 y 0.8624.

2. **GraphSAGE superó consistentemente a GCN.**
   Esto sugiere que el esquema de agregación vecinal de GraphSAGE fue más adecuado para los grafos textuales construidos a partir de similitud TF-IDF y k-NN.

3. **La utilidad de las GNN en NLP depende de la calidad del grafo.**
   Cuando la señal léxica ya es muy fuerte, como en estos experimentos, un modelo clásico puede seguir siendo la alternativa más eficiente. Las GNN tienden a ser más convenientes cuando las relaciones entre instancias aportan información adicional que no está tan explícita en el texto crudo.

En síntesis, el grafo **sí ayudó a estructurar el espacio de representación**, como se aprecia en t-SNE y en la estabilidad de GraphSAGE, pero **no produjo una mejora cuantitativa sobre el baseline** en los dos casos evaluados.

Visto desde la finalidad del proyecto, esto no invalida la propuesta: más bien delimita con mayor precisión **qué solución es actualmente más conveniente para resolver el problema planteado**. Hoy, SentinelGrafo muestra que un sistema inteligente para clasificación y priorización de texto puede ser funcional y útil, pero que la capa gráfica todavía necesita optimización adicional para justificar su adopción frente a métodos más simples en un entorno real.


## 9. Conclusiones

A partir del desarrollo completo del informe, se concluye que SentinelGrafo cumplió satisfactoriamente con el propósito central del proyecto: **investigar las Redes Neuronales Gráficas y traducir ese conocimiento en un software inteligente funcional** orientado a la clasificación y priorización de información textual. El trabajo no solo abordó los fundamentos teóricos de las GNN y su comparación con otras arquitecturas, sino que además los llevó a una implementación concreta, evaluable y alineada con un caso de uso real.

Desde la perspectiva del informe en su conjunto, también puede afirmarse que el proyecto permitió articular de manera coherente sus distintas partes: el marco teórico justificó la elección de GraphSAGE y GCN; la sección de desarrollo mostró cómo se construyó el pipeline de limpieza, vectorización y generación de grafos; los experimentos computacionales validaron el comportamiento de los modelos; y la demo interactiva evidenció que la propuesta puede presentarse como un sistema utilizable y no solo como código aislado. En ese sentido, SentinelGrafo sí constituye un **prototipo funcional de software inteligente de baja complejidad**.

Más importante aún, el trabajo se alinea con la finalidad de construir **software inteligente que resuelva un problema concreto**. En este caso, la problemática abordada es la necesidad de clasificar grandes volúmenes de mensajes o noticias para identificar información relevante, prioritaria o potencialmente crítica, reduciendo la carga de revisión manual y mejorando la capacidad de respuesta del usuario o analista. Bajo esa lógica, SentinelGrafo debe entenderse como una solución de apoyo a tareas de monitoreo, filtrado y priorización de contenido textual en escenarios donde revisar manualmente toda la información sería poco eficiente.

En el plano técnico, la evidencia obtenida muestra que **las GNN fueron viables y competitivas**, pero no superaron al baseline de Logistic Regression. Por ello, la principal lección del estudio es que **incorporar estructura de grafo no garantiza por sí mismo un mejor rendimiento**; el beneficio real depende de cuánta información relacional adicional aporta el grafo frente a representaciones textuales ya muy fuertes. Esto no debilita el proyecto, sino que lo vuelve más valioso: permite concluir con criterio cuándo una solución basada en grafos aporta ventajas reales y cuándo un enfoque clásico puede seguir siendo más conveniente para un despliegue inicial.

También se concluye que **GraphSAGE fue la arquitectura más sólida dentro de las GNN evaluadas**. En ambos casos obtuvo mejores métricas que GCN, mostró mayor estabilidad en entrenamiento y generó embeddings visualmente más organizados. Esto la convierte en la opción preferible si el proyecto se extendiera manteniendo la misma lógica de construcción del grafo. En términos de ingeniería del software, esta conclusión ayuda a decidir qué arquitectura conviene priorizar si SentinelGrafo evolucionara hacia una versión más robusta o más cercana a producción.

A nivel de aporte general del informe, el proyecto también deja una conclusión metodológica importante: para construir soluciones de IA útiles no basta con elegir una técnica avanzada; es necesario relacionarla con el problema, justificar su diseño, implementarla correctamente, compararla con alternativas razonables y evaluar sus resultados con criterio. En ese sentido, el informe demuestra que la investigación exploratoria sí se transformó en una solución concreta, analizada y sustentada.

Entre las limitaciones del trabajo destacan las siguientes:

- El grafo se construyó únicamente a partir de similitud coseno sobre TF-IDF, por lo que la calidad estructural depende directamente de esa representación.
- Solo se evaluaron dos arquitecturas GNN y una configuración fija de hiperparámetros.
- El entrenamiento se realizó sobre CPU y con *early stopping*, lo que favorece la factibilidad experimental, pero restringe la exploración de configuraciones más profundas o costosas.
- En el Caso 1 persiste una sensibilidad limitada para detectar la clase de desastre real, lo cual es relevante si el sistema se pensara para escenarios críticos.

Como trabajo futuro, sería recomendable:

- construir grafos usando embeddings semánticos más ricos, por ejemplo SBERT o modelos tipo Transformer;
- experimentar con **GAT/GATv2** u otras variantes que ponderen mejor la importancia de cada vecino;
- ajustar de manera sistemática parámetros como **k**, número de capas, dimensión oculta y estrategia de regularización;
- incorporar grafos heterogéneos o relaciones adicionales, como hashtags, usuarios, entidades o coocurrencias temporales.

En conclusión, SentinelGrafo demostró que las Redes Neuronales Gráficas son una herramienta válida para modelar problemas de NLP con estructura relacional, pero en estos experimentos el mejor equilibrio entre simplicidad y rendimiento siguió estando del lado del baseline clásico. El valor principal del proyecto, por tanto, no solo estuvo en la métrica final, sino en evidenciar **cuándo un enfoque gráfico aporta y cuándo un método tradicional sigue siendo suficiente** dentro de una solución orientada a resolver un problema real.

Bajo esa interpretación, el proyecto no debe verse solo como un ejercicio demostrativo, sino como un **prototipo funcional de software inteligente** para clasificación y priorización de información textual. Su aporte real está en mostrar una ruta concreta de solución: recibir texto, representarlo, clasificarlo y convertirlo en una salida útil para apoyar decisiones en escenarios de monitoreo, vigilancia digital o análisis de contenido crítico. Por ello, las conclusiones del informe no se limitan a decir qué modelo ganó, sino que muestran que el proyecto logró integrar teoría, implementación, experimentación y utilidad práctica en una misma propuesta.
